In [ ]:
# import os
# import pandas as pd

# # Set the path to the folder containing the downloaded files
# download_dir = r'C:\Users\terbe\Desktop\Folder and Files Archiving -Box Sdk\temp_downloads'

# # Initialize list to hold file metadata
# file_data = []

# # Loop through all files in the directory
# for root, _, files in os.walk(download_dir):
#     for file in files:
#         full_path = os.path.join(root, file)
#         file_name = os.path.basename(file)
#         file_id = os.path.splitext(file_name)[0].split("_")[-1]  # extract the last underscore-separated part
#         extension = os.path.splitext(file_name)[1].replace('.', '')  # get extension without the dot
#         file_data.append({
#             'file_path': full_path,
#             'file_name': file_name,
#             'file_id': file_id,
#             'extension': extension
#         })

# # Create a DataFrame
# df_files = pd.DataFrame(file_data)

# #!pip install pywin32

# import os
# import subprocess
# import tempfile
# import shutil
# import pandas as pd
# from docx import Document

# # for COM fallback
# import pythoncom
# import win32com.client

# # 1) Define exactly the extensions you care about
# doc_exts = ['doc', 'docx', 'doc1', 'doc2', 'doc5', 'do']

# # 2) Filter your DataFrame
# df_docs = df_files[df_files['extension'].isin(doc_exts)].reset_index(drop=True)

# def extract_doc_text(path: str) -> str:
#     """
#     Extract plain text from Word‐style docs on Windows:
#       - .docx via python-docx
#       - .doc* via antiword (if installed)
#       - fallback to MS Word COM automation
#     """
#     ext = os.path.splitext(path)[1].lower().lstrip('.')
    
#     # --- DOCX: pure python
#     if ext == 'docx':
#         try:
#             doc = Document(path)
#             return '\n'.join(p.text for p in doc.paragraphs)
#         except Exception as e:
#             return f"[error reading .docx: {e}]"
    
#     # --- DOC variants: try antiword first
#     if ext in {'doc', 'doc1', 'doc2', 'doc5', 'do'}:
#         # 1) antiword
#         try:
#             raw = subprocess.check_output(
#                 ['antiword', path],
#                 stderr=subprocess.DEVNULL
#             )
#             return raw.decode('utf-8', errors='replace')
#         except Exception:
#             pass
        
#         # 2) fallback to MS Word COM
#         try:
#             pythoncom.CoInitialize()              # initialize COM in this thread
#             word = win32com.client.Dispatch("Word.Application")
#             word.Visible = False
#             doc = word.Documents.Open(path, ReadOnly=True)
#             text = doc.Content.Text
#             doc.Close(False)
#             word.Quit()
#             pythoncom.CoUninitialize()
#             return text
#         except Exception as e:
#             return f"[error via COM automation: {e}]"
    
#     return "[unsupported extension]"

# # 3) Apply extraction
# df_docs['document_text'] = (
#     df_docs['file_path']
#     .apply(lambda p: extract_doc_text(p) if os.path.exists(p) else "[file missing]")
# )

# # 4) Inspect results
# print(df_docs.columns)
# print(df_docs['extension'].value_counts())
# print(df_docs[['file_name','extension','document_text']].head())

# df_docs.to_csv(r"C:\Users\terbe\Desktop\Folder and Files Archiving -Box Sdk\parquet_batches\documents_text_extraction_complete.csv")

In [ ]:
# import pandas as pd

# # Path to the file you just wrote
# docs_csv = r"C:\Users\terbe\Desktop\Folder and Files Archiving -Box Sdk\parquet_batches\documents_text_extraction_complete.csv"

# # Read it back in
# df_docs_loaded = pd.read_csv(docs_csv)

# # Quick sanity check
# print("Rows × Columns:", df_docs_loaded.shape)
# print(df_docs_loaded.head())


# Excel Files

In [ ]:
# import os
# import pandas as pd
# from win32com.client import Dispatch

# # 1) Folder of downloaded files
# download_dir = r'C:\Users\terbe\Desktop\Folder and Files Archiving -Box Sdk\temp_downloads'

# # 2) Walk and collect metadata
# file_data = []
# for root, _, files in os.walk(download_dir):
#     for fname in files:
#         full_path = os.path.join(root, fname)
#         base, ext = os.path.splitext(fname)
#         ext = ext.lstrip('.').lower()
#         file_id = base.split('_')[-1]
#         file_data.append({
#             'file_path': full_path,
#             'file_name': fname,
#             'file_id': file_id,
#             'extension': ext
#         })
# df_files = pd.DataFrame(file_data)

# # 3) Keep only Excel-style files
# excel_exts = ['xls', 'xls2', 'xlsx']
# df_excels = df_files[df_files['extension'].isin(excel_exts)].reset_index(drop=True)

# # 4) COM fallback functions
# def _com_sheets(path: str):
#     """Yield (sheet_name, [header_values]) for each sheet via Excel COM."""
#     excel = Dispatch('Excel.Application')
#     wb = excel.Workbooks.Open(path, ReadOnly=True)
#     for sheet in wb.Sheets:
#         used = sheet.UsedRange
#         headers = []
#         for col in range(1, used.Columns.Count + 1):
#             val = sheet.Cells(1, col).Value
#             if val is not None:
#                 headers.append(str(val))
#         yield sheet.Name, headers
#     wb.Close(False)
#     excel.Quit()

# def excel_information(path: str) -> str:
#     try:
#         parts = []
#         flat_seen = []
#         for name, headers in _com_sheets(path):
#             parts.append(f"Sheet '{name}': {', '.join(headers)}")
#             for h in headers:
#                 if h not in flat_seen:
#                     flat_seen.append(h)
#         flat_str = ", ".join(flat_seen)
#         return "\n".join(parts) + "\nFlat headers: " + flat_str
#     except Exception as e:
#         return f"[failed to read Excel via COM: {e}]"

# # 5) Build the combined column
# df_excels['excel_information'] = df_excels['file_path'].apply(
#     lambda p: excel_information(p) if os.path.exists(p) else "[file missing]"
# )

# # 6) If you want to drop the old cols and just keep this:
# df_result = df_excels[[
#     'file_path','file_name','file_id','extension','excel_information'
# ]]
# df_result.to_csv(r"C:\Users\terbe\Desktop\Folder and Files Archiving -Box Sdk\excel_llama_summaries.csv")



In [ ]:
# import os
# import pandas as pd
# import subprocess
# from pathlib import Path

# # ─── CONFIG ──────────────────────────────────────────────────────────────────
# OLLAMA_PATH = r"C:\Users\terbe\AppData\Local\Programs\Ollama\ollama.exe"
# MODEL_NAME  = "llama3.2"
# OUTPUT_CSV  = Path(r"C:\Users\terbe\Desktop\Folder and Files Archiving -Box Sdk\excel_llama_summaries.csv")
# BATCH_SIZE  = 3

# # ─── LOAD your existing results (ensures df == df_summaries) ─────────────────
# df = pd.read_csv(OUTPUT_CSV, index_col=0)

# # Make sure the summary column exists
# if 'excel_summary' not in df.columns:
#     df['excel_summary'] = ""

# # ─── DEFINE LLaMA CALL ────────────────────────────────────────────────────────

# def llama_guess(headers: str) -> str:
#     prompt = (
#         "I have a spreadsheet with these columns and information:\n\n"
#         f"{headers}\n\n"
#         "Based on those column names, please write a quick summary of what this excel document contains."
#     )
#     proc = subprocess.Popen(
#         [OLLAMA_PATH, "run", MODEL_NAME],
#         stdin=subprocess.PIPE,
#         stdout=subprocess.PIPE,
#         stderr=subprocess.PIPE,
#     )
#     # send UTF-8, get raw bytes back
#     out_bytes, err_bytes = proc.communicate(input=prompt.encode('utf-8'))
#     # decode with utf-8, replacing any invalid bytes
#     text = out_bytes.decode('utf-8', errors='replace').strip()
#     if not text:
#         return "[no response]"
#     return text

# # ─── FIND ROWS STILL TO PROCESS ───────────────────────────────────────────────
# pending = df[df['excel_summary'].isna() | (df['excel_summary'] == "")].index.tolist()

# # ─── PROCESS IN BATCHES ───────────────────────────────────────────────────────
# for batch_start in range(0, len(pending), BATCH_SIZE):
#     batch_idxs = pending[batch_start: batch_start + BATCH_SIZE]
#     for idx in batch_idxs:
#         info = df.at[idx, 'excel_information']
#         try:
#             df.at[idx, 'excel_summary'] = llama_guess(info)
#         except Exception as e:
#             df.at[idx, 'excel_summary'] = f"[ERROR: {e}]"
#         print(f"Row {idx} done.")

#     # Save after each batch (with index) so you can resume without Unnamed: 0
#     df.to_csv(OUTPUT_CSV, index=True)
#     print(f"Saved batch {batch_start//BATCH_SIZE + 1} of {((len(pending)-1)//BATCH_SIZE)+1}.")

# print("All done! Summaries are in", OUTPUT_CSV)


In [ ]:
import pandas as pd

excel_df = pd.read_csv(r"C:\Users\terbe\Desktop\Folder and Files Archiving -Box Sdk\excel_llama_summaries.csv")
excel_df

# Powerpoints 

In [ ]:
# import os
# import pandas as pd

# # Set the path to the folder containing the downloaded files
# download_dir = r'C:\Users\terbe\Desktop\Folder and Files Archiving -Box Sdk\temp_downloads'

# # Initialize list to hold file metadata
# file_data = []

# # Loop through all files in the directory
# for root, _, files in os.walk(download_dir):
#     for file in files:
#         full_path = os.path.join(root, file)
#         file_name = os.path.basename(file)
#         file_id = os.path.splitext(file_name)[0].split("_")[-1]  # extract the last underscore-separated part
#         extension = os.path.splitext(file_name)[1].replace('.', '')  # get extension without the dot
#         file_data.append({
#             'file_path': full_path,
#             'file_name': file_name,
#             'file_id': file_id,
#             'extension': extension
#         })

# # Create a DataFrame
# df_files = pd.DataFrame(file_data)

In [ ]:
import os
import re
import pythoncom
import win32com.client
from pptx import Presentation
import pandas as pd

# --- 0) Prepare your DataFrame of PowerPoint files ---
# assume df_files already exists and has columns "file_path", "file_name", "extension"
powerpoint_df = df_files[df_files["extension"].isin(["ppt", "ppt1", "pptx"])].copy()

# --- 1) Initialize COM and launch PowerPoint once ---
pythoncom.CoInitialize()
pp_app = win32com.client.DispatchEx("PowerPoint.Application")
# minimize the window so it doesn’t get in your way
try:
    pp_app.WindowState = 2  # 2 == ppWindowMinimized
except Exception:
    pass

def extract_ppt_text(path: str) -> str:
    """
    Extracts all text from a .pptx file via python-pptx,
    or from .ppt/.ppt1 via PowerPoint COM.
    Returns a single string (slides separated by newlines),
    or an error message.
    """
    ext = os.path.splitext(path)[1].lower().lstrip('.')
    texts = []

    if ext == "pptx":
        try:
            prs = Presentation(path)
            for slide in prs.slides:
                for shape in slide.shapes:
                    if hasattr(shape, "text") and shape.text.strip():
                        texts.append(shape.text.strip())
        except Exception as e:
            return f"[error reading .pptx: {e}]"

    elif ext in ("ppt", "ppt1"):
        try:
            # positional args: Open(path, ReadOnly, Untitled, WithWindow)
            pres = pp_app.Presentations.Open(path, 1, 0, 0)
            for slide in pres.Slides:
                for shape in slide.Shapes:
                    if shape.HasTextFrame and shape.TextFrame.HasText:
                        txt = shape.TextFrame.TextRange.Text.strip()
                        if txt:
                            texts.append(txt)
            pres.Close()
        except Exception as e:
            return f"[error reading .ppt via COM: {e}]"
    else:
        return "[unsupported extension]"

    return "\n".join(texts) if texts else "[no text found]"

# --- 2) Extract all texts ---
powerpoint_df["ppt_text"] = powerpoint_df["file_path"].apply(
    lambda p: extract_ppt_text(p) if os.path.exists(p) else "[file missing]"
)

# --- 3) Compute sentence and token counts ---
def count_sentences(text: str) -> int:
    # split on punctuation followed by space
    parts = re.split(r'(?<=[.!?])\s+', text)
    return len([s for s in parts if s.strip()])

powerpoint_df["sentence_count"] = powerpoint_df["ppt_text"].apply(count_sentences)
powerpoint_df["token_count"]    = powerpoint_df["ppt_text"].apply(lambda t: len(t.split()))


# --- 5) Clean up COM ---
pp_app.Quit()
pythoncom.CoUninitialize()



In [ ]:
import re
import pandas as pd
from nltk.tokenize import sent_tokenize
from sentence_transformers import SentenceTransformer, util

# --- Load your PPT DataFrame (assumes ppt_text column already exists) ---
# powerpoint_df = ...

# --- Load the sentence-transformer model once ---
model = SentenceTransformer('all-MiniLM-L6-v2')

# --- Text cleaning and counting helpers ---
def clean_text(text):
    return re.sub(r'\s+', ' ', text).strip() if isinstance(text, str) else ""

def count_tokens(text):
    return len(text.split()) if isinstance(text, str) else 0

def count_sentences(text):
    return len(sent_tokenize(text)) if isinstance(text, str) and text.strip() else 0

# --- Semantic summarization to top-N sentences ---
def summarize_text(text, max_sentences=250):
    sentences = sent_tokenize(text)
    if len(sentences) <= max_sentences:
        return text
    # embed and score
    embeddings = model.encode(sentences, convert_to_tensor=True)
    scores = util.pytorch_cos_sim(embeddings, embeddings).mean(dim=1)
    top_idxs = scores.argsort(descending=True)[:max_sentences]
    # return in original order
    return " ".join([sentences[i] for i in sorted(top_idxs.tolist())])

# --- Conditional summarization based on thresholds ---
def conditional_summarize(row,
                          max_sentences=250,
                          max_tokens=5000):
    text = row['ppt_text']
    clean = clean_text(text)
    sents = count_sentences(clean)
    toks  = count_tokens(clean)
    if sents > max_sentences or toks > max_tokens:
        # summarize by sentences first
        summary = summarize_text(clean, max_sentences=max_sentences)
        # if still too many tokens, truncate to max_tokens
        tokens = summary.split()
        if len(tokens) > max_tokens:
            summary = " ".join(tokens[:max_tokens])
        return summary
    else:
        return clean

# --- Apply to your DataFrame ---
powerpoint_df['clean_text']              = powerpoint_df['ppt_text'].apply(clean_text)
powerpoint_df['sentence_count']          = powerpoint_df['clean_text'].apply(count_sentences)
powerpoint_df['token_count']             = powerpoint_df['clean_text'].apply(count_tokens)
powerpoint_df['ppt_summarized']          = powerpoint_df.apply(conditional_summarize, axis=1)
powerpoint_df['summarized_sentence_count'] = powerpoint_df['ppt_summarized'].apply(count_sentences)
powerpoint_df['summarized_token_count']    = powerpoint_df['ppt_summarized'].apply(count_tokens)

powerpoint_df.reset_index(drop=True, inplace=True)
powerpoint_df.to_csv(
    r"C:\Users\terbe\Desktop\Folder and Files Archiving -Box Sdk\powerpoint_df_with_summaries.csv",
    index=False,
    encoding="utf-8-sig"   # include BOM for better Excel compatibility
)
powerpoint_df.reset_index(drop=True, inplace=True)
powerpoint_df['ppt_summarized'][0]

In [ ]:
import pandas as pd

ppt_df = pd.read_csv(r"C:\Users\terbe\Desktop\Folder and Files Archiving -Box Sdk\powerpoint_df_with_summaries.csv")
ppt_df

In [ ]:
import subprocess
import pandas as pd
import os

# --- Setup ---
ollama_path = r"C:\Users\terbe\AppData\Local\Programs\Ollama\ollama.exe"
model_name = "llama3.2"
input_csv_path = r"C:\Users\terbe\Desktop\Folder and Files Archiving -Box Sdk\powerpoint_df_with_summaries.csv"
output_parquet_path = r"C:\Users\terbe\Desktop\Folder and Files Archiving -Box Sdk\powerpoint_df_with_llama_summaries.csv"
batch_size = 3

# --- Define function to call Llama 3.2 model ---
def call_llama_summarize(text):
    prompt = f"Summarize this text:\n\n{text}"

    result = subprocess.run(
        [ollama_path, "run", model_name],
        input=prompt,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
        encoding="utf-8"
    )

    if result.returncode != 0:
        print("Error:", result.stderr)
        return ""

    return result.stdout.strip()


# --- Load or create dataframe ---
if os.path.exists(output_parquet_path):
    df_pdf = pd.read_parquet(output_parquet_path)
    print("Loaded existing processed file.")
else:
    df_pdf = pd.read_csv(input_csv_path)
    df_pdf["ppt_llama_summary"] = ""
    df_pdf["processed"] = False
    print("Loaded fresh input CSV.")

# --- Ensure required columns ---
for col in ["ppt_llama_summary", "processed"]:
    if col not in df_pdf.columns:
        df_pdf[col] = "" if col == "summary" else False

# --- Fix rows that have no text: mark them processed ---
for idx, row in df_pdf[df_pdf["processed"] == False].iterrows():
    text = row["ppt_summarized"]
    if pd.isna(text) or not isinstance(text, str) or text.strip() == "":
        df_pdf.at[idx, "processed"] = True

# --- Find where to resume ---
start_idx = df_pdf[df_pdf["processed"] == False].index.min()
if pd.isna(start_idx):
    print("All rows already processed.")
    start_idx = len(df_pdf)
else:
    print(f"Resuming from index {start_idx}.")

# --- Processing Loop ---
batch_counter = 0

for idx in range(start_idx, len(df_pdf)):
    row = df_pdf.loc[idx]
    text_to_summarize = row["ppt_summarized"]

    if pd.notna(text_to_summarize) and isinstance(text_to_summarize, str) and text_to_summarize.strip():
        summary = call_llama_summarize(text_to_summarize)
        df_pdf.at[idx, "ppt_llama_summary"] = summary
        df_pdf.at[idx, "processed"] = True
        batch_counter += 1

    # Save after every batch_size rows
    if batch_counter >= batch_size:
        df_pdf.to_parquet(output_parquet_path, index=False)
        print(f"Saved after processing up to index {idx}.")
        batch_counter = 0

# --- Final save ---
df_pdf.to_parquet(output_parquet_path, index=False)
print("Final save complete.")


# Emails 

In [ ]:
.eml, mbox	Email files and archives	email module, extract_msg, or mailbox

In [ ]:
import os
import pandas as pd

# Set the path to the folder containing the downloaded files
download_dir = r'C:\Users\terbe\Desktop\Folder and Files Archiving -Box Sdk\temp_downloads'

# Initialize list to hold file metadata
file_data = []

# Loop through all files in the directory
for root, _, files in os.walk(download_dir):
    for file in files:
        full_path = os.path.join(root, file)
        file_name = os.path.basename(file)
        file_id = os.path.splitext(file_name)[0].split("_")[-1]  # extract the last underscore-separated part
        extension = os.path.splitext(file_name)[1].replace('.', '')  # get extension without the dot
        file_data.append({
            'file_path': full_path,
            'file_name': file_name,
            'file_id': file_id,
            'extension': extension
        })

# Create a DataFrame
df_files = pd.DataFrame(file_data)

emls_df =df_files[df_files["extension"].isin(["eml", "mbox"])]
emls_df


file_path	file_name	file_id	extension
3	C:\Users\terbe\Desktop\Folder and Files Archiv...	06311_1844506238339.eml	1844506238339	eml
47	C:\Users\terbe\Desktop\Folder and Files Archiv...	2008_1844515969267.eml	1844515969267	eml
62	C:\Users\terbe\Desktop\Folder and Files Archiv...	2009_1844516410902.eml	1844516410902	eml
63	C:\Users\terbe\Desktop\Folder and Files Archiv...	2014_1844502467222.eml	1844502467222	eml
454	C:\Users\terbe\Desktop\Folder and Files Archiv...	last_1844502602013.eml	1844502602013	eml
467	C:\Users\terbe\Desktop\Folder and Files Archiv...	mbox_1844517222093.eml	1844517222093	eml
526	C:\Users\terbe\Desktop\Folder and Files Archiv...	RecoveredEML_1844493179900.mbox	1844493179900	mbox
672	C:\Users\terbe\Desktop\Folder and Files Archiv...	~$coveredEML_1844492764640.mbox	1844492764640	mbox

    
from email import policy
from email.parser import BytesParser
from bs4 import BeautifulSoup
import os

i need to iterate through all files in this dataframe so I can save each txt file, then parse it, then save each
parsed email in in its own rows in email_df, saving and concatenating all of the emails from each file, please
store and save the original file_name and file_path  when parsing, so that we know the oriignal source, but we will
need to save each txt file, parse, and concatenate all email_df 

# Path to your .eml file
eml_path = emls_df["file_path"][i]

import csv

output_path = r"C:\Users\terbe\Desktop\file_name.txt" # please change each output path with the same name as the original file name emls_df["file_name"], like this 
# like this output_path = r"C:\Users\terbe\Desktop\emls_df["file_name"][i].txt"
output_csv = r"C:\Users\terbe\Desktop\parsed_file_name.csv" # please put the respective file name her from which was parsed. emls_df["file_name"]
# like this output_path = r"C:\Users\terbe\Desktop\parsed_emls_df["file_name"][i].csv"

# Prepare the CSV output
with open(output_csv, 'w', newline='', encoding='utf-8') as out_csv:
    writer = csv.DictWriter(out_csv, fieldnames=["Subject", "From", "To", "Date", "Body"])
    writer.writeheader()

    current_email = {"subject": "", "from": "", "to": "", "date": "", "body": []}

    def write_current_email():
        if any(current_email.values()):
            writer.writerow({
                "Subject": current_email["subject"],
                "From": current_email["from"],
                "To": current_email["to"],
                "Date": current_email["date"],
                "Body": "\n".join(current_email["body"]).strip()
            })

    with open(output_path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()

            if line.startswith("Subject:"):
                write_current_email()
                current_email = {"subject": line[8:].strip(), "from": "", "to": "", "date": "", "body": []}
            elif line.startswith("From:"):
                current_email["from"] = line[5:].strip()
            elif line.startswith("To:"):
                current_email["to"] = line[3:].strip()
            elif line.startswith("Date:"):
                current_email["date"] = line[5:].strip()
            else:
                current_email["body"].append(line)

    write_current_email()  # write final one

add the original filename in a emails_df["original_file_name"]
for each iteration concatenate the emails_df together,    
pasrsed emails should be concatenated and saved 
please also print which file_name we are currently on - and which email row we are currently on

In [ ]:
# Set the path to the folder containing the downloaded files
download_dir = r'C:\Users\terbe\Desktop\Folder and Files Archiving -Box Sdk\emails to text'

# Initialize list to hold file metadata
file_data = []

# Loop through all files in the directory
for root, _, files in os.walk(download_dir):
    for file in files:
        full_path = os.path.join(root, file)
        file_name = os.path.basename(file)
        file_id = os.path.splitext(file_name)[0].split("_")[-1]  # extract the last underscore-separated part
        extension = os.path.splitext(file_name)[1].replace('.', '')  # get extension without the dot
        file_data.append({
            'file_path': full_path,
            'file_name': file_name,
            'file_id': file_id,
            'extension': extension
        })

# Create a DataFrame
email_text_df = pd.DataFrame(file_data)
email_text_df




In [ ]:
import os
import pandas as pd
import csv

# Input DataFrame: email_text_df
parsed_dir = r"C:\Users\terbe\Desktop\Folder and Files Archiving -Box Sdk\parsed_emails"  # Optional: store all CSVs in a subfolder
os.makedirs(parsed_dir, exist_ok=True)

for idx, row in email_text_df.iterrows():
    input_txt_path = row["file_path"]
    file_name_no_ext = os.path.splitext(row["file_name"])[0]
    output_csv_path = os.path.join(parsed_dir, f"parsed_{file_name_no_ext}.csv")

    print(f"📄 Parsing {row['file_name']} ({idx + 1} of {len(email_text_df)})")

    try:
        with open(output_csv_path, 'w', newline='', encoding='utf-8') as out_csv:
            writer = csv.DictWriter(out_csv, fieldnames=["Subject", "From", "To", "Date", "Body", "Original_File"])
            writer.writeheader()

            current_email = {"subject": "", "from": "", "to": "", "date": "", "body": []}

            def write_current_email():
                if any(current_email.values()):
                    writer.writerow({
                        "Subject": current_email["subject"],
                        "From": current_email["from"],
                        "To": current_email["to"],
                        "Date": current_email["date"],
                        "Body": "\n".join(current_email["body"]).strip(),
                        "Original_File": row["file_name"]
                    })

            with open(input_txt_path, 'r', encoding='utf-8') as f:
                for line in f:
                    line = line.strip()

                    if line.startswith("Subject:"):
                        write_current_email()
                        current_email = {"subject": line[8:].strip(), "from": "", "to": "", "date": "", "body": []}
                    elif line.startswith("From:"):
                        current_email["from"] = line[5:].strip()
                    elif line.startswith("To:"):
                        current_email["to"] = line[3:].strip()
                    elif line.startswith("Date:"):
                        current_email["date"] = line[5:].strip()
                    else:
                        current_email["body"].append(line)

            write_current_email()  # write last email

        print(f"✅ Saved to: {output_csv_path}")

    except Exception as e:
        print(f"❌ Error processing {row['file_name']}: {e}")


In [ ]:
import pandas as pd

file_path = r"C:\Users\terbe\Desktop\Folder and Files Archiving -Box Sdk\parsed_emails\parsed_06311_1844506238339.csv"

# Use python engine and skip bad lines
df_one = pd.read_csv(file_path, engine='python', on_bad_lines='skip')

# Preview
print(df_one.head())
print(df_one.columns)


In [ ]:
import os
import pandas as pd

# Directory where the parsed CSVs are stored
parsed_dir = r"C:\Users\terbe\Desktop\Folder and Files Archiving -Box Sdk\parsed_emails"

# List all parsed CSV files
csv_files = [f for f in os.listdir(parsed_dir) if f.endswith('.csv')]

# Dictionary to store each DataFrame keyed by filename (without extension)
email_dfs = {}
all_dfs = []  # for merging

for csv_file in csv_files:
    file_path = os.path.join(parsed_dir, csv_file)
    try:
        df = pd.read_csv(file_path, engine="python", on_bad_lines="skip")
        key = os.path.splitext(csv_file)[0]
        email_dfs[key] = df
        all_dfs.append(df)
        print(f"✅ Loaded: {key} with {len(df)} rows")
    except Exception as e:
        print(f"❌ Failed to load {csv_file}: {e}")

# Merge all DataFrames
combined_df = pd.concat(all_dfs, ignore_index=True)
print(f"\n📦 Combined total: {len(combined_df)} rows from {len(all_dfs)} files")

# Save the combined CSV
output_path = os.path.join(parsed_dir, r"C:\Users\terbe\Desktop\Folder and Files Archiving -Box Sdk\combined_parsed_emails.csv")
combined_df.to_csv(output_path, index=False)
print(f"✅ Saved combined DataFrame to:\n{output_path}")


✅ Loaded: parsed_06311_1844506238339 with 8563386 rows
✅ Loaded: parsed_2008_1844515969267 with 18203517 rows
✅ Loaded: parsed_2009_1844516410902 with 20936249 rows
✅ Loaded: parsed_2014_1844502467222 with 4656815 rows
✅ Loaded: parsed_last_1844502602013 with 14579012 rows
✅ Loaded: parsed_mbox_1844517222093 with 9371312 rows
✅ Loaded: parsed_output_email_text with 8563386 rows
✅ Loaded: parsed_RecoveredEML_1844493179900 with 35032 rows
✅ Loaded: parsed_~$coveredEML_1844492764640 with 1 rows

📦 Combined total: 84908710 rows from 9 files


OSError: [Errno 28] No space left on device

In [ ]:
import pandas as pd

# Correct full path to the saved file
combined_path = r"C:\Users\terbe\Desktop\Folder and Files Archiving -Box Sdk\parsed_emails\combined_parsed_emails.csv"

# Read the file back in
combined_df = pd.read_csv(combined_path, engine='python', on_bad_lines='skip')

combined_df

# Microsoft Access DB
.eml, mbox	Email files and archives	email module, extract_msg, or mailbox
.mdb	Microsoft Access DB	pyodbc or mdbtools


# SQL
.db	SQLite databases	sqlite3, SQL queries


In [ ]:
import os
import pandas as pd

# Set the path to the folder containing the downloaded files
download_dir = r'C:\Users\terbe\Desktop\Folder and Files Archiving -Box Sdk\temp_downloads'

# Initialize list to hold file metadata
file_data = []

# Loop through all files in the directory
for root, _, files in os.walk(download_dir):
    for file in files:
        full_path = os.path.join(root, file)
        file_name = os.path.basename(file)
        file_id = os.path.splitext(file_name)[0].split("_")[-1]  # extract the last underscore-separated part
        extension = os.path.splitext(file_name)[1].replace('.', '')  # get extension without the dot
        file_data.append({
            'file_path': full_path,
            'file_name': file_name,
            'file_id': file_id,
            'extension': extension
        })

# Create a DataFrame
df_files = pd.DataFrame(file_data)

In [ ]:
# List the “SQL database” extensions you care about
sql_exts = ['mdb', 'db']

# Create the sub‐DataFrame
df_sql = df_files[df_files['extension'].isin(sql_exts)].reset_index(drop=True)

# Inspect
print(df_sql)


# HTML
.mhtml, .mht, .htm	Archived or plain HTML	BeautifulSoup or html2text


In [ ]:
# 1. Define the HTML extensions you care about
html_exts = ['html', 'mhtml', 'mht', 'htm']

# 2. Filter df_files to just those rows
df_html = df_files[df_files['extension'].isin(html_exts)].reset_index(drop=True)

# 3. (Optional) Inspect the result
print(df_html)


In [ ]:
df_html.columns == Index(['file_path', 'file_name', 'file_id', 'extension'], dtype='object')

In [ ]:
from bs4 import BeautifulSoup

def extract_html_text(path):
    # read the raw HTML
    with open(path, 'r', encoding='utf-8', errors='ignore') as f:
        html = f.read()
    # parse and pull only the visible text
    soup = BeautifulSoup(html, 'html.parser')
    return soup.get_text(separator=' ', strip=True)

# apply to each file_path and store in a new column
df_html['text'] = df_html['file_path'].apply(extract_html_text)

# optional: peek at what you got
print(df_html[['file_name', 'text']].head())


                                           file_name  \
0                          2820007_1844496433932.htm   
1  Don't_Tell_Me_by_Lisa_Janicke_Hinchliffe..._-_...   
2  Expanding_Public_Colleges_in_the_Trump_Era_184...   
3  Freedom_Forum__Toward_Black_Liberation_in_the_...   
4           GWS_Solidarity_Hours_1844530679022.mhtml   

                                                text  
0  CHICAGO HILTON 720 South Michigan Avenue Chica...  
1  From: Subject: Don't Tell Me by Lisa Janicke H...  
2  From: Subject: Expanding Public Colleges in th...  
3  From: Subject: Freedom Forum: Toward Black Lib...  
4  From: Subject: GWS Solidarity Hours\nDate: Thu...  


In [ ]:
# 1. Define the HTML extensions you care about
html_exts = ['html', 'mhtml', 'mht', 'htm']

# 2. Filter df_files to just those rows
df_html = df_files[df_files['extension'].isin(html_exts)].reset_index(drop=True)

# 3. (Optional) Inspect the result
print(df_html)


from bs4 import BeautifulSoup

def extract_html_text(path):
    # read the raw HTML
    with open(path, 'r', encoding='utf-8', errors='ignore') as f:
        html = f.read()
    # parse and pull only the visible text
    soup = BeautifulSoup(html, 'html.parser')
    return soup.get_text(separator=' ', strip=True)

# apply to each file_path and store in a new column
df_html['text'] = df_html['file_path'].apply(extract_html_text)

# optional: peek at what you got
print(df_html[['file_name', 'text']].head())


import re
import subprocess
import pandas as pd
from nltk.tokenize import sent_tokenize
from sentence_transformers import SentenceTransformer, util

# ——— Settings ———
MAX_SENTS   = 250
MAX_TOKENS  = 5000
OLLAMA_EXE  = r"C:\Users\terbe\AppData\Local\Programs\Ollama\ollama.exe"
OLLAMA_MODEL= "llama3.2"

# ——— Load / copy your HTML‐text DF ———
# assume df_html already has a column `text` with your scraped HTML
html_df = df_html.copy()

# ——— (Optional) sentence‐transformer model ———
st_model = SentenceTransformer("all-MiniLM-L6-v2")

# ——— Helpers ———
def clean_text(s: str) -> str:
    return re.sub(r"\s+", " ", s).strip()

def conditional_doc_summary(s: str) -> str:
    txt = clean_text(s)
    sents = sent_tokenize(txt)
    if len(sents) > MAX_SENTS or len(txt.split()) > MAX_TOKENS:
        # embed & score sentences
        emb = st_model.encode(sents, convert_to_tensor=True)
        scores = util.pytorch_cos_sim(emb, emb).mean(dim=1)
        topn = scores.argsort(descending=True)[:MAX_SENTS]
        picked = [sents[i] for i in sorted(topn.tolist())]
        summ = " ".join(picked)
        toks = summ.split()
        return " ".join(toks[:MAX_TOKENS])
    return txt

def llama_summarize(s: str) -> str:
    proc = subprocess.run(
        [OLLAMA_EXE, "run", OLLAMA_MODEL],
        input=f"Summarize this text:\n\n{s}",
        stdout=subprocess.PIPE, stderr=subprocess.PIPE,
        text=True, encoding="utf-8"
    )
    if proc.returncode != 0:
        raise RuntimeError(f"Ollama error: {proc.stderr}")
    return proc.stdout.strip()

# ——— Apply to your DF ———
# 1) create the pre-Llama summary (original or clipped)
html_df["cond_summary"] = html_df["text"].apply(conditional_doc_summary)

# 2) feed that into Ollama for the final summary
html_df["llama_summary"] = html_df["cond_summary"].apply(llama_summarize)

# ——— (Optional) save out ———
output_csv = r"C:\Users\terbe\Desktop\Folder and Files Archiving -Box Sdk\html_with_summaries.csv"
html_df.to_csv(output_csv, index=False, encoding="utf-8-sig")
print("Wrote:", output_csv)


Wrote: C:\Users\terbe\Desktop\Folder and Files Archiving -Box Sdk\html_with_summaries.csv


# Videos
.mov	Video files	Use ffmpeg to extract audio/subtitles or take screenshots + OCR


In [ ]:
# 1. Define the video extensions you care about
video_exts = ['mov']

# 2. Filter df_files to just those rows
df_videos = df_files[df_files['extension'].isin(video_exts)].reset_index(drop=True)

# 3. (Optional) Inspect the result
print(df_videos)


                                           file_path  \
0  C:\Users\terbe\Desktop\Folder and Files Archiv...   
1  C:\Users\terbe\Desktop\Folder and Files Archiv...   

                    file_name        file_id extension  
0  IMG_1644_1844530091499.mov  1844530091499       mov  
1  MVI_4306_1844530955687.mov  1844530955687       mov  


# Text
.txt	Plaintext	Direct read via open()

In [ ]:
import os
import pandas as pd

# Set the path to the folder containing the downloaded files
download_dir = r'C:\Users\terbe\Desktop\Folder and Files Archiving -Box Sdk\temp_downloads'

# Initialize list to hold file metadata
file_data = []

# Loop through all files in the directory
for root, _, files in os.walk(download_dir):
    for file in files:
        full_path = os.path.join(root, file)
        file_name = os.path.basename(file)
        file_id = os.path.splitext(file_name)[0].split("_")[-1]  # extract the last underscore-separated part
        extension = os.path.splitext(file_name)[1].replace('.', '')  # get extension without the dot
        file_data.append({
            'file_path': full_path,
            'file_name': file_name,
            'file_id': file_id,
            'extension': extension
        })

# Create a DataFrame
df_files = pd.DataFrame(file_data)

# 1. Define the text extensions you care about
text_exts = ['txt']

# 2. Filter df_files to just those rows
df_text = df_files[df_files['extension'].isin(text_exts)].reset_index(drop=True)

# define a helper to read in a text file (ignore any encoding errors)
def _read_txt(path):
    with open(path, 'r', encoding='utf-8', errors='ignore') as f:
        return f.read()

# create the new “Text” column by applying to each file_path
df_text['Text'] = df_text['file_path'].apply(_read_txt)

df_text



                                           file_path  \
0  C:\Users\terbe\Desktop\Folder and Files Archiv...   
1  C:\Users\terbe\Desktop\Folder and Files Archiv...   

                      file_name        file_id extension  
0    6607Ref1_1844494096586.txt  1844494096586       txt  
1  FM20_11312_1844489835764.txt  1844489835764       txt  


In [ ]:
# define a helper to read in a text file (ignore any encoding errors)
def _read_txt(path):
    with open(path, 'r', encoding='utf-8', errors='ignore') as f:
        return f.read()

# create the new “Text” column by applying to each file_path
df_text['Text'] = df_text['file_path'].apply(_read_txt)

df_text


# RTF 
.rtf	Rich Text Format	pypandoc or convert to .txt using unrtf

In [ ]:
# 1. Define the RTF extension you care about
rtf_exts = ['rtf']

# 2. Filter df_files to just those rows
df_rtf = df_files[df_files['extension'].isin(rtf_exts)].reset_index(drop=True)

# 3. (Optional) Inspect the result
print(df_rtf)


# ZIPS
.zip	Compressed archives	Use zipfile to extract contents
.gz	Compressed files	Use gzip module to decompress, then read


In [ ]:
# 1. Define the compressed‐archive extensions you care about
zip_exts = ['zip', 'gz']

# 2. Filter df_files to just those rows
df_archives = df_files[df_files['extension'].isin(zip_exts)].reset_index(drop=True)

# 3. (Optional) Inspect the result
print(df_archives)


# Handle Unknowns 
.toc, d1, d, career2, hanrttydoc, italics, edu'sconflictedcopy2011-12-09), lagrangian, particleturbulence, 	Unknown/custom labels	Try file content inspection or ignore if unreadable
00361-00850, 00136-01590, 00106-00790, 00111-00480, 00701-01750, 00701-01000	Likely custom batch labels	May refer to scanned pages—try OCR or skip
net3852448d, net0e594322, dmdelive13b254ad	Corrupted or malformed	Try filename cleaning + guessing format

In [ ]:
# 1. Define the list of “known” extensions (from your initial table)
known_exts = [
    'pdf', 'jpg', 'png', 'gif', 'tif', 'eps',
    'doc', 'docx',
    'xls', 'xlsx',
    'ppt', 'ppt1',
    'eml', 'mbox',
    'mdb', 'db',
    'mhtml', 'mht', 'htm',
    'mov',
    'rtf',
    'zip', 'gz',
    'txt'
]

# 2. Filter out anything that's not in that list
df_unknown = df_files[~df_files['extension'].isin(known_exts)].reset_index(drop=True)

# 3. (Optional) Inspect which “unknown” extensions you have
print(df_unknown['extension'].unique())

# 4. Now df_unknown contains:
#    • Custom/unknown labels like .toc, d1, career2, etc.
#    • Batch-style labels like 00361-00850, 00136-01590, …
#    • Corrupted/malformed ones like net3852448d, dmdelive13b254ad, …
# You can then decide whether to OCR, skip, or further clean based on df_unknown.


                                           file_path  \
0  C:\Users\terbe\Desktop\Folder and Files Archiv...   
1  C:\Users\terbe\Desktop\Folder and Files Archiv...   
2  C:\Users\terbe\Desktop\Folder and Files Archiv...   
3  C:\Users\terbe\Desktop\Folder and Files Archiv...   

                            file_name        file_id extension  
0     9781107041202_1844489250229.zip  1844489250229       zip  
1      fig1002.tar-1_1844491446906.gz  1844491446906        gz  
2        fig1002.tar_1844487299117.gz  1844487299117        gz  
3  hanratty_mito.tar_1844496395832.gz  1844496395832        gz  


In [ ]:
!pip install striprtf


In [ ]:
import os
import re
import subprocess
import pandas as pd
from nltk.tokenize import sent_tokenize
from striprtf.striprtf import rtf_to_text  # 🡐 pure-Python RTF→text

# 1) (Optional) coarse summarization
HAVE_ST = False
try:
    from sentence_transformers import SentenceTransformer, util
    st_model = SentenceTransformer('all-MiniLM-L6-v2')
    HAVE_ST = True
except ImportError:
    pass

def clean_text(t):
    return re.sub(r'\s+', ' ', t).strip()

def conditional_summarize(t, max_sents=250, max_toks=5000):
    txt = clean_text(t)
    if HAVE_ST:
        sents = sent_tokenize(txt)
        if len(sents) > max_sents or len(txt.split()) > max_toks:
            embeds = st_model.encode(sents, convert_to_tensor=True)
            scores = util.pytorch_cos_sim(embeds, embeds).mean(dim=1)
            topn = scores.argsort(descending=True)[:max_sents]
            sel = " ".join([sents[i] for i in sorted(topn.tolist())])
            toks = sel.split()
            return " ".join(toks[:max_toks]) if len(toks) > max_toks else sel
    return txt

# 2) Ollama summarization
OLLAMA_EXE = r"C:\Users\terbe\AppData\Local\Programs\Ollama\ollama.exe"
MODEL_NAME  = "llama3.2"

def llama_summarize(text):
    result = subprocess.run(
        [OLLAMA_EXE, "run", MODEL_NAME],
        input=f"Summarize this text:\n\n{text}",
        stdout=subprocess.PIPE, stderr=subprocess.PIPE,
        text=True, encoding="utf-8"
    )
    if result.returncode != 0:
        raise RuntimeError(result.stderr)
    return result.stdout.strip()

# 3) Paths and batching
DOWNLOAD_DIR = r'C:\Users\terbe\Desktop\Folder and Files Archiving -Box Sdk\temp_downloads'
OUTPUT_CSV   = r'C:\Users\terbe\Desktop\Folder and Files Archiving -Box Sdk\rtf_text_df_with_llama_summaries.csv'
BATCH_SIZE   = 3

# 4) Build the DataFrame of RTF files
rows = []
for root, _, files in os.walk(DOWNLOAD_DIR):
    for fn in files:
        if fn.lower().endswith('.rtf'):
            rows.append({
                'file_path': os.path.join(root, fn),
                'file_name': fn
            })
rtf_df = pd.DataFrame(rows)

# 5) Convert to text
def _read_rtf(path):
    raw = open(path, 'r', encoding='utf-8', errors='ignore').read()
    return rtf_to_text(raw)

rtf_df['Text'] = rtf_df['file_path'].apply(_read_rtf)

# 6) Pre-summarize if needed
rtf_df['pre_summary'] = rtf_df['Text'].apply(conditional_summarize)

# 7) Final Ollama summaries
rtf_df['llama_summary'] = ""
rtf_df['processed']     = False
batch_ctr = 0

for i in rtf_df.index:
    chunk = rtf_df.at[i, 'pre_summary']
    if chunk:
        rtf_df.at[i, 'llama_summary'] = llama_summarize(chunk)
    rtf_df.at[i, 'processed'] = True
    batch_ctr += 1

    if batch_ctr >= BATCH_SIZE:
        rtf_df.to_csv(OUTPUT_CSV, index=False, encoding='utf-8-sig')
        batch_ctr = 0

# 8) Final save
rtf_df.to_csv(OUTPUT_CSV, index=False, encoding='utf-8-sig')
print("Done. Saved to:", OUTPUT_CSV)


Done. Saved to: C:\Users\terbe\Desktop\Folder and Files Archiving -Box Sdk\rtf_text_df_with_llama_summaries.csv


In [ ]:
import pandas as pd


# Path to the saved CSV
output_csv_path = r'C:\Users\terbe\Desktop\Folder and Files Archiving -Box Sdk\rtf_text_df_with_llama_summaries.csv'

# Read it back in
df_rtf = pd.read_csv(output_csv_path, encoding="utf-8-sig")
df_rtf


In [ ]:
df_rtf["pre_summary"][0]

In [ ]:
df_rtf["llama_summary"][0]

In [ ]:
import pandas as pd

final_df = pd.read_csv(
    r"C:\Users\terbe\Desktop\Folder and Files Archiving -Box Sdk\final_df_summaries.csv"
)

# quick sanity check
print(final_df.shape)
final_df.head()


(554, 19)


In [ ]:
for i in range(500):
    print(final_df["llama_generated_summary_from_truncated_text"][i])

In [ ]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
import faiss
import pickle

# ─── 1) Load your LLAMA summaries ─────────────────────────────
df = pd.read_csv(
    r"C:\Users\terbe\Desktop\Folder and Files Archiving -Box Sdk\final_df_summaries.csv"
)
# ensure no NaNs
summaries = df['llama_generated_summary_from_truncated_text'].fillna("").tolist()

# ─── 2) Compute embeddings ──────────────────────────────────────
model = SentenceTransformer("all-MiniLM-L6-v2")
embs = model.encode(
    summaries,
    convert_to_numpy=True,
    show_progress_bar=True
).astype('float32')

# ─── 3) Build a HNSWFAISS index ────────────────────────────────
d = embs.shape[1]           # embedding dimension
M = 32                      # HNSW connectivity parameter
index = faiss.IndexHNSWFlat(d, M)
index.hnsw.efConstruction = 200  # controls build time / accuracy
index.add(embs)
index.hnsw.efSearch = 50       # controls query-time accuracy/speed

# ─── 4) Persist index + metadata ──────────────────────────────
faiss.write_index(index, "summaries_hnsw.index")

# we'll save only the columns you need to display on hits
meta = df[['file_path','file_name','file_id','extension','llama_generated_summary_from_truncated_text']]
with open("summaries_metadata.pkl", "wb") as f:
    pickle.dump(meta, f)

print("✅ Built & saved FAISS index and metadata.")

# ─── 5) Query helper ───────────────────────────────────────────
def search_summaries(query: str, top_k: int = 5):
    """
    Returns a list of dicts with keys:
    - file_id, file_name, file_path, distance, summary
    """
    # encode query
    q_emb = model.encode([query], convert_to_numpy=True).astype('float32')
    # search
    D, I = index.search(q_emb, top_k)

    # load metadata
    with open("summaries_metadata.pkl","rb") as f:
        meta = pickle.load(f)

    results = []
    for dist, idx in zip(D[0], I[0]):
        row = meta.iloc[idx]
        results.append({
            "file_id":       row.file_id,
            "file_name":     row.file_name,
            "file_path":     row.file_path,
            "distance":      float(dist),
            "summary":       row.llama_generated_summary_from_truncated_text
        })
    return results

# ─── Example ───────────────────────────────────────────────────
if __name__ == "__main__":
    hits = search_summaries("campus research grants", top_k=3)
    for hit in hits:
        print(f"{hit['distance']:.4f} — {hit['file_name']}")
        print("  >", hit['summary'], "\n")
